In [1]:
import re
import json
import pandas as pd
from spacy import displacy
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# ============================================================
# 1. LOAD SEMUA MODEL
# ============================================================
print("Loading models...")

# Model POS Tagger
pos_tokenizer = AutoTokenizer.from_pretrained("w11wo/indonesian-roberta-base-posp-tagger")
pos_model     = AutoModelForTokenClassification.from_pretrained("w11wo/indonesian-roberta-base-posp-tagger")
pos_pipeline  = pipeline("token-classification", model=pos_model, tokenizer=pos_tokenizer,
                         aggregation_strategy="simple")

# Model NER (aggregated untuk displaCy)
ner_tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-NER")
ner_model     = AutoModelForTokenClassification.from_pretrained("cahya/bert-base-indonesian-NER")
ner_pipeline  = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer,
                         aggregation_strategy="simple")

# Model NER (raw untuk IOB tags)
ner_pipeline_raw = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer)

print("Semua model berhasil dimuat!\n")

# ============================================================
# 2. FUNGSI PREPROCESSING
# ============================================================
def clean_text(text):
    text = re.sub(r'\d+\s+(?:[A-Za-z]\s+){2,}[A-Za-z]+', '', text)
    text = re.sub(r'Kumpulan Dongeng nusantara\s+\d+', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        words = line.strip().split()
        if len(words) > 3 or len(words) == 0:
            cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

# ============================================================
# 3. FUNGSI CHUNKING DENGAN OFFSET + OVERLAP
# ============================================================
def chunk_text_with_offset(text, max_length=400, overlap=50):
    """
    Memotong teks menjadi chunks dengan:
    - max_length : batas karakter per chunk
    - overlap    : jumlah karakter yang diulang antar chunk
                   agar entitas di batas chunk tidak terpotong
    Mengembalikan list of (chunk_text, start_offset)
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks        = []
    current       = ""
    current_start = 0
    pointer       = 0

    for sent in sentences:
        sent_start = text.find(sent, pointer)

        if len(current) + len(sent) < max_length:
            if current == "":
                current_start = sent_start
            current += (" " + sent) if current else sent
        else:
            if current:
                chunks.append((current.strip(), current_start))
                # Ambil overlap dari akhir chunk sebelumnya
                overlap_text  = current[-overlap:] if len(current) > overlap else current
                overlap_start = current_start + len(current) - len(overlap_text)
                current       = overlap_text + " " + sent
                current_start = overlap_start
            else:
                current       = sent
                current_start = sent_start

        pointer = sent_start + len(sent)

    if current:
        chunks.append((current.strip(), current_start))

    return chunks  # [(chunk_text, start_offset), ...]

# ============================================================
# 4. FUNGSI TOKENISASI (word-level)
# ============================================================
def tokenize(text):
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return tokens

# ============================================================
# 5. FUNGSI NOUN PHRASE GROUPING DARI HASIL POS TAGS
# ============================================================
NOUN_TAGS = {"NN", "NNP", "NNS", "NNPS"}

def extract_noun_phrases(pos_results):
    noun_phrases = []
    current_np   = []
    for token in pos_results:
        tag  = token["entity_group"]
        word = token["word"]
        if tag in NOUN_TAGS:
            current_np.append(word)
        else:
            if current_np:
                noun_phrases.append(" ".join(current_np))
                current_np = []
    if current_np:
        noun_phrases.append(" ".join(current_np))
    return noun_phrases

# ============================================================
# 6. FUNGSI DEDUPLIKASI ENTITAS (akibat overlap antar chunk)
# ============================================================
def deduplicate_entities(entities):
    """
    Hilangkan entitas duplikat yang muncul akibat overlap chunk.
    Prioritaskan entitas dengan score lebih tinggi.
    Dua entitas dianggap duplikat jika posisinya overlap.
    """
    if not entities:
        return []

    # Urutkan berdasarkan start position
    sorted_ents = sorted(entities, key=lambda x: x["start"])
    deduped     = [sorted_ents[0]]

    for current in sorted_ents[1:]:
        last = deduped[-1]
        # Cek apakah posisi overlap dengan entitas terakhir
        if current["start"] < last["end"]:
            # Ambil yang score-nya lebih tinggi
            if current["score"] > last["score"]:
                deduped[-1] = current
        else:
            deduped.append(current)

    return deduped

# ============================================================
# 7. FUNGSI KONVERSI KE FORMAT DISPLACY
# ============================================================
def to_displacy_format(text, entities):
    ents = []
    for ent in entities:
        ents.append({
            "start": ent["start"],
            "end":   ent["end"],
            "label": ent["entity_group"]
        })
    return {"text": text, "ents": ents, "title": None}

# ============================================================
# 8. LOAD DATASET
# ============================================================
with open(r"C:\Semester 4\NLP\Pembelajaran Bahasa Mendalam\Tugas NER\data\dataset_ner_200.json", encoding="utf-8") as f:
    data = json.load(f)

print(f"Dataset berhasil dimuat: {len(data)} dokumen\n")

# ============================================================
# 9. PIPELINE UTAMA — PROSES SEMUA DOKUMEN
# ============================================================
all_tokens       = []
all_pos_results  = []
all_noun_phrases = []
all_iob_rows     = []
all_aggregated   = []  # per dokumen (bukan per chunk)

print("Memproses semua dokumen...")

for doc_idx, item in enumerate(data):

    # --- Cleaning ---
    teks = clean_text(item["isi"])

    # --- Chunking dengan offset & overlap ---
    chunks_with_offset = chunk_text_with_offset(teks, max_length=400, overlap=50)

    doc_entities_raw = []  # kumpulkan semua entitas sebelum deduplikasi

    for chunk_idx, (chunk, offset) in enumerate(chunks_with_offset):

        # Step 1 — Tokenisasi
        tokens = tokenize(chunk)
        all_tokens.append({
            "doc_idx":   doc_idx,
            "chunk_idx": chunk_idx,
            "tokens":    tokens,
            "jumlah":    len(tokens)
        })

        # Step 2 — POS Tagging
        pos_results = pos_pipeline(chunk)
        for pos in pos_results:
            all_pos_results.append({
                "doc_idx":   doc_idx,
                "chunk_idx": chunk_idx,
                "word":      pos["word"],
                "pos_tag":   pos["entity_group"],
                "score":     round(pos["score"], 4)
            })

        # Step 3 — Noun Phrase Grouping
        noun_phrases = extract_noun_phrases(pos_results)
        for np_text in noun_phrases:
            all_noun_phrases.append({
                "doc_idx":     doc_idx,
                "chunk_idx":   chunk_idx,
                "noun_phrase": np_text
            })

        # Step 4 — IOB Tags (raw NER)
        raw_entities = ner_pipeline_raw(chunk)
        for ent in raw_entities:
            all_iob_rows.append({
                "doc_idx":   doc_idx,
                "chunk_idx": chunk_idx,
                "word":      ent["word"],
                "iob_tag":   ent["entity"],
                "score":     round(ent["score"], 4)
            })

        # Step 5 — Named Entity Extraction + koreksi offset
        agg_entities = ner_pipeline(chunk)
        for ent in agg_entities:
            doc_entities_raw.append({
                "start":        ent["start"] + offset,
                "end":          ent["end"]   + offset,
                "entity_group": ent["entity_group"],
                "word":         ent["word"],
                "score":        round(ent["score"], 4)
            })

    # Deduplikasi entitas hasil overlap antar chunk
    doc_entities_final = deduplicate_entities(doc_entities_raw)

    # Simpan per dokumen (teks utuh + entitas final)
    all_aggregated.append({
        "doc_idx":  doc_idx,
        "teks":     teks,
        "entities": doc_entities_final
    })

    if (doc_idx + 1) % 20 == 0:
        print(f"  {doc_idx + 1}/{len(data)} dokumen selesai...")

print(f"\nSelesai! Total dokumen: {len(all_aggregated)}\n")

# ============================================================
# 10. KONVERSI KE DATAFRAME
# ============================================================
df_tokens = pd.DataFrame(all_tokens)
df_pos    = pd.DataFrame(all_pos_results)
df_np     = pd.DataFrame(all_noun_phrases)
df_iob    = pd.DataFrame(all_iob_rows)

rekap = []
for item in all_aggregated:
    for ent in item["entities"]:
        rekap.append({
            "doc_idx": item["doc_idx"],
            "entity":  ent["word"],
            "label":   ent["entity_group"],
            "score":   ent["score"]
        })
df_rekap = pd.DataFrame(rekap)

# ============================================================
# 11. TAMPILKAN HASIL PER LANGKAH (DOKUMEN 0)
# ============================================================
print("=" * 60)
print("LANGKAH 1 — TOKENISASI (Dokumen 0, Chunk 0)")
print("=" * 60)
doc0_tok = df_tokens[(df_tokens["doc_idx"] == 0) & (df_tokens["chunk_idx"] == 0)]
print(f"Jumlah token : {doc0_tok['jumlah'].values[0]}")
print(f"Contoh token : {doc0_tok['tokens'].values[0][:20]} ...")

print("\n" + "=" * 60)
print("LANGKAH 2 — POS TAGGING (Dokumen 0)")
print("=" * 60)
df_pos_doc0 = df_pos[df_pos["doc_idx"] == 0].reset_index(drop=True)
display(df_pos_doc0.head(30))

print("\n" + "=" * 60)
print("LANGKAH 3 — NOUN PHRASE GROUPING (Dokumen 0)")
print("=" * 60)
df_np_doc0 = df_np[df_np["doc_idx"] == 0].reset_index(drop=True)
display(df_np_doc0.head(20))

print("\n" + "=" * 60)
print("LANGKAH 4 — IOB TAGS (Dokumen 0, hanya non-O)")
print("=" * 60)
df_iob_doc0 = df_iob[
    (df_iob["doc_idx"] == 0) & (df_iob["iob_tag"] != "O")
].reset_index(drop=True)
display(df_iob_doc0.head(30))

print("\n" + "=" * 60)
print("LANGKAH 5 — NAMED ENTITIES + VISUALISASI DISPLACY (Dokumen 0)")
print("=" * 60)

colors = {
    "PER": "#f9a825",
    "LOC": "#42a5f5",
    "ORG": "#66bb6a",
}

doc0      = all_aggregated[0]
doc_data  = to_displacy_format(doc0["teks"], doc0["entities"])
displacy.render(doc_data, style="ent", manual=True, jupyter=True,
                options={"colors": colors})

# ============================================================
# 12. ANALISIS TAMBAHAN
# ============================================================
print("\n" + "=" * 60)
print("ANALISIS — DISTRIBUSI POS TAGS (semua dokumen)")
print("=" * 60)
display(df_pos["pos_tag"].value_counts()
        .reset_index()
        .rename(columns={"index": "pos_tag", "pos_tag": "frekuensi"}))

print("\n" + "=" * 60)
print("ANALISIS — FREKUENSI LABEL NER (semua dokumen)")
print("=" * 60)
display(df_rekap["label"].value_counts())

print("\n" + "=" * 60)
print("ANALISIS — ENTITAS PALING SERING MUNCUL (Top 20)")
print("=" * 60)
display(df_rekap.groupby(["entity", "label"]).size()
        .reset_index(name="frekuensi")
        .sort_values("frekuensi", ascending=False)
        .head(20))

print("\n" + "=" * 60)
print("ANALISIS — PERBANDINGAN NOUN PHRASES vs NAMED ENTITIES")
print("=" * 60)
total_np  = len(df_np)
total_ner = len(df_rekap)
np_set    = set(df_np["noun_phrase"].str.lower())
ner_set   = set(df_rekap["entity"].str.lower())
overlap   = np_set & ner_set

print(f"Total Noun Phrases ditemukan  : {total_np}")
print(f"Total Named Entities ditemukan: {total_ner}")
print(f"Overlap (NP yang juga jadi NER): {len(overlap)}")
print(f"Coverage : {round(len(overlap) / len(ner_set) * 100, 2)}% NER tercakup dalam Noun Phrases")

Loading models...


Device set to use cpu
Some weights of the model checkpoint at cahya/bert-base-indonesian-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Semua model berhasil dimuat!

Dataset berhasil dimuat: 200 dokumen

Memproses semua dokumen...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  20/200 dokumen selesai...
  40/200 dokumen selesai...
  60/200 dokumen selesai...
  80/200 dokumen selesai...
  100/200 dokumen selesai...
  120/200 dokumen selesai...
  140/200 dokumen selesai...
  160/200 dokumen selesai...
  180/200 dokumen selesai...
  200/200 dokumen selesai...

Selesai! Total dokumen: 200

LANGKAH 1 — TOKENISASI (Dokumen 0, Chunk 0)
Jumlah token : 64
Contoh token : ['Lilo', 'adalah', 'nyamuk', 'jantan', 'kecil', 'yang', 'cerdas', '.', 'Dia', 'dan', 'keluarganya', 'tinggal', 'di', 'Kolam', 'Tamansari', 'Kerajaan', 'Ardiloka', '.', 'Kerajaan', 'Ardiloka'] ...

LANGKAH 2 — POS TAGGING (Dokumen 0)


...


ANALISIS — FREKUENSI LABEL NER (semua dokumen)


label
PER    7812
LOC     967
PRD     836
CRD     723
ORG     579
GPE     561
NOR     167
QTY     123
WOA     112
REG     103
EVT      92
ORD      57
DAT      27
TIM      11
FAC       9
LAN       5
LAW       2
MON       1
Name: count, dtype: int64


ANALISIS — ENTITAS PALING SERING MUNCUL (Top 20)


,entity,label,frekuensi
1646,kedua,CRD,141
3028,satu,CRD,117
406,aji,PER,110
1084,dua,CRD,106
1800,kiki,PER,79
1045,dimas,PER,66
1444,jaka bagus,PER,65
3445,tuhan,REG,59
1585,kanc,PER,56
3236,sumarni,PER,53



ANALISIS — PERBANDINGAN NOUN PHRASES vs NAMED ENTITIES
Total Noun Phrases ditemukan  : 50555
Total Named Entities ditemukan: 12187
Overlap (NP yang juga jadi NER): 111
Coverage : 3.5% NER tercakup dalam Noun Phrases
